# LightGBM Model

In [8]:
# =============================================================================
# LightGBM MODEL — FULL PIXEL — CHIRPS + ERA5 (features) -> CDI (target)
# =============================================================================
# WHY LightGBM:
#   - Leaf-wise tree growth (vs level-wise in XGBoost) -> higher accuracy
#   - Histogram-based algorithm -> 10x faster on large pixel datasets
#   - Built-in early stopping, feature importance, and cross-validation
#   - Consistently top performer on tabular/raster regression tasks
# OUTPUT: lgbm_fullpixel_results_v1/
# =============================================================================

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb

# =============================================================================
# GLOBAL FONT SIZE
# =============================================================================

FS = 25

plt.rcParams.update({
    "font.size":        FS,
    "axes.titlesize":   FS,
    "axes.labelsize":   FS,
    "xtick.labelsize":  FS,
    "ytick.labelsize":  FS,
    "legend.fontsize":  FS,
    "figure.titlesize": FS,
    "font.family":      "serif",
    "axes.linewidth":   1.2,
})

SEED = 42
np.random.seed(SEED)

# =============================================================================
# PATHS
# =============================================================================

BASE_DIR   = r"C:\Users\gerii\Documents\projects\tekeze"
CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_resampled")
ERA5_DIR   = os.path.join(BASE_DIR, "ERA5_resampled")
CDI_DIR    = os.path.join(BASE_DIR, "indices", "CDI_resampled")

output_dir = os.path.join(BASE_DIR, "lgbm_fullpixel_results_v1")
os.makedirs(output_dir, exist_ok=True)

print("=" * 70)
print("LightGBM FULL-PIXEL MODEL — CHIRPS + ERA5 -> CDI")
print("=" * 70)

# =============================================================================
# STEP 1: FIND COMMON MONTHS
# =============================================================================

def get_keys(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.tif")))
    return set(os.path.basename(f).replace(".tif", "") for f in files)

chirps_keys = get_keys(CHIRPS_DIR)
era5_keys   = get_keys(ERA5_DIR)
cdi_keys    = get_keys(CDI_DIR)
common_keys = sorted(chirps_keys & era5_keys & cdi_keys)

print(f"\n  CHIRPS months : {len(chirps_keys)}")
print(f"  ERA5   months : {len(era5_keys)}")
print(f"  CDI    months : {len(cdi_keys)}")
print(f"  COMMON months : {len(common_keys)}")
print(f"  Period        : {common_keys[0]} to {common_keys[-1]}")

# =============================================================================
# STEP 2: LOAD RASTERS — with month index as extra feature
# =============================================================================
# BONUS: we add month_of_year (1-12) as a third feature to capture
# seasonality — this alone often boosts R² by 0.03-0.05 for CDI

def load_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arr[arr <= -9999] = np.nan
    return arr

print("\nLoading and pooling all pixels across common months ...")

precip_list, tmean_list, month_list, cdi_list = [], [], [], []

for i, key in enumerate(common_keys):
    chirps_path = os.path.join(CHIRPS_DIR, f"{key}.tif")
    era5_path   = os.path.join(ERA5_DIR,   f"{key}.tif")
    cdi_path    = os.path.join(CDI_DIR,    f"{key}.tif")

    # extract month number from key e.g. "2003_07" -> 7
    month_num = int(key.split("_")[1])

    precip_arr = load_band(chirps_path)
    tmean_arr  = load_band(era5_path)
    cdi_arr    = load_band(cdi_path)

    if not (precip_arr.shape == tmean_arr.shape == cdi_arr.shape):
        print(f"  WARNING: shape mismatch at {key} — skipping")
        continue

    precip_flat = precip_arr.flatten()
    tmean_flat  = tmean_arr.flatten()
    cdi_flat    = cdi_arr.flatten()

    valid = (~np.isnan(precip_flat)) & (~np.isnan(tmean_flat)) & (~np.isnan(cdi_flat))
    n_valid = valid.sum()

    precip_list.append(precip_flat[valid])
    tmean_list.append(tmean_flat[valid])
    month_list.append(np.full(n_valid, month_num, dtype=np.float32))
    cdi_list.append(cdi_flat[valid])

    if (i + 1) % 50 == 0 or (i + 1) == len(common_keys):
        print(f"  Processed {i + 1}/{len(common_keys)} months ...")

precip_all = np.concatenate(precip_list)
tmean_all  = np.concatenate(tmean_list)
month_all  = np.concatenate(month_list)
cdi_all    = np.concatenate(cdi_list)

print(f"\n  Total pooled samples: {len(precip_all):,}")

df = pd.DataFrame({
    "precip": precip_all,
    "tmean":  tmean_all,
    "month":  month_all,   # seasonal feature
    "cdi":    cdi_all
})

print(df.describe().round(3))

# Features: precip + tmean + month_of_year
FEATURE_COLS = ["precip", "tmean", "month"]

X = df[FEATURE_COLS].values
y = df["cdi"].values

print(f"\n  X shape : {X.shape}  (features: {FEATURE_COLS})")
print(f"  y shape : {y.shape}")

# =============================================================================
# TRAIN / TEST SPLIT — 70 / 30
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, shuffle=True
)

print(f"\n  Training samples : {len(X_train):,}")
print(f"  Testing  samples : {len(X_test):,}")

# LightGBM doesn't need scaling but keep for consistency
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# =============================================================================
# METRICS
# =============================================================================

def nash_sutcliffe(obs, sim):
    return 1.0 - (np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2))

def pbias(obs, sim):
    return 100.0 * np.sum(obs - sim) / np.sum(obs)

def compute_metrics(obs, pred, tag):
    rmse = np.sqrt(mean_squared_error(obs, pred))
    mae  = mean_absolute_error(obs, pred)
    r2   = r2_score(obs, pred)
    nse  = nash_sutcliffe(obs, pred)
    pb   = pbias(obs, pred)
    print(f"  {tag} -> R2={r2:.4f}  RMSE={rmse:.4f}  "
          f"MAE={mae:.4f}  NSE={nse:.4f}  PBIAS={pb:.2f}%")
    return r2, rmse, mae, nse, pb

# =============================================================================
# LightGBM DATASET
# =============================================================================

lgb_train = lgb.Dataset(X_train_sc, label=y_train,
                         feature_name=FEATURE_COLS, free_raw_data=False)
lgb_test  = lgb.Dataset(X_test_sc,  label=y_test,
                         reference=lgb_train, free_raw_data=False)

# =============================================================================
# HYPERPARAMETERS
# =============================================================================

params = {
    "objective":        "regression",
    "metric":           "rmse",
    "boosting_type":    "gbdt",          # gradient boosting decision tree
    "num_leaves":       127,             # key param: controls model complexity
    "max_depth":        -1,              # no limit (controlled by num_leaves)
    "learning_rate":    0.05,
    "n_estimators":     1000,            # max trees (early stopping cuts this)
    "subsample":        0.8,             # row subsampling
    "subsample_freq":   1,
    "colsample_bytree": 1.0,             # all 3 features
    "reg_alpha":        0.01,            # L1
    "reg_lambda":       1.0,             # L2
    "min_child_samples": 20,
    "verbose":          -1,
    "seed":             SEED,
    "n_jobs":           -1,
}

# =============================================================================
# TRAIN WITH EARLY STOPPING
# =============================================================================

print("\nTraining LightGBM ...")

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
]

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_test],
    valid_names=["train", "test"],
    callbacks=callbacks,
)

print(f"\n  Best iteration : {model.best_iteration}")

# =============================================================================
# SAVE MODEL
# =============================================================================

model.save_model(os.path.join(output_dir, "lgbm_model_fullpixel_v1.txt"))
print("Model saved.")

# =============================================================================
# PREDICTIONS AND METRICS
# =============================================================================

print("\nGenerating predictions ...")
y_train_pred = model.predict(X_train_sc, num_iteration=model.best_iteration)
y_test_pred  = model.predict(X_test_sc,  num_iteration=model.best_iteration)

print("\nLightGBM Results:")
tr = compute_metrics(y_train, y_train_pred, "Train")
te = compute_metrics(y_test,  y_test_pred,  "Test ")

gap = tr[0] - te[0]

pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "R2":      [tr[0], te[0]],
    "RMSE":    [tr[1], te[1]],
    "MAE":     [tr[2], te[2]],
    "NSE":     [tr[3], te[3]],
    "PBIAS":   [tr[4], te[4]],
}).to_csv(os.path.join(output_dir, "train_test_metrics.csv"), index=False)

pd.DataFrame({"Observed_CDI": y_train, "Predicted_CDI": y_train_pred}).to_csv(
    os.path.join(output_dir, "train_predictions.csv"), index=False)
pd.DataFrame({"Observed_CDI": y_test, "Predicted_CDI": y_test_pred}).to_csv(
    os.path.join(output_dir, "test_predictions.csv"), index=False)

# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================

fi = pd.DataFrame({
    "Feature":    FEATURE_COLS,
    "Importance": model.feature_importance(importance_type="gain"),
}).sort_values("Importance", ascending=False)

print(f"\n  Feature Importances (gain):\n{fi.to_string(index=False)}")
fi.to_csv(os.path.join(output_dir, "feature_importance.csv"), index=False)

# Feature importance bar plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi["Feature"], fi["Importance"], color=["#2166ac", "#d7191c", "#1a9641"])
ax.set_xlabel("Importance (Gain)")
ax.set_title("LightGBM Feature Importance")
ax.invert_yaxis()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, "feature_importance.png"),
            dpi=300, bbox_inches="tight")
plt.close(fig)
print("Saved: feature_importance.png")

# =============================================================================
# FIGURE 1: TWO-PANEL SCATTER
# =============================================================================

def axis_lims(a, b, pad=0.05):
    lo  = min(a.min(), b.min())
    hi  = max(a.max(), b.max())
    rng = hi - lo
    return lo - pad * rng, hi + pad * rng

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

vmin = min(y_train_pred.min(), y_test_pred.min())
vmax = max(y_train_pred.max(), y_test_pred.max())

panel_data = [
    (y_train, y_train_pred, tr[0], tr[1], tr[2], "(a)"),
    (y_test,  y_test_pred,  te[0], te[1], te[2], "(b)"),
]

for ax, (obs, pred, r2_val, rmse_val, mae_val, label) in zip(axes, panel_data):
    lo, hi = axis_lims(obs, pred)

    sc = ax.scatter(obs, pred, alpha=0.25, s=14,
                    c=pred, cmap="RdYlBu_r",
                    vmin=vmin, vmax=vmax,
                    edgecolors="none", rasterized=True)

    ax.plot([lo, hi], [lo, hi], color="black", linewidth=2,
            linestyle="--", label="1:1 line")

    m, b = np.polyfit(obs, pred, 1)
    x_fit = np.linspace(lo, hi, 200)
    ax.plot(x_fit, m * x_fit + b, color="#d7191c", linewidth=2.0,
            label=f"Fit (slope={m:.2f})")

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Observed CDI")
    ax.set_ylabel("Predicted (LightGBM) CDI")
    ax.set_title(label)

    stats_text = (
        f"$R^2$ = {r2_val:.2f}\n"
        f"RMSE = {rmse_val:.2f}\n"
        f"MAE  = {mae_val:.2f}"
    )
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
            fontsize=FS - 1, verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                      edgecolor="grey", alpha=0.85))

    ax.legend(loc="lower right", framealpha=0.85)
    ax.grid(True, linestyle="--", alpha=0.4, linewidth=0.8)

plt.tight_layout()
cbar_ax = fig.add_axes([1.02, 0.15, 0.02, 0.70])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label("CDI", fontsize=FS)
cbar.ax.tick_params(labelsize=FS - 1)
fig.savefig(os.path.join(output_dir, "obs_vs_pred_combined.png"),
            dpi=400, bbox_inches="tight")
plt.close(fig)
print("Saved: obs_vs_pred_combined.png")

# =============================================================================
# FIGURE 2: LOSS CURVE (training RMSE per boosting round)
# =============================================================================

# For lgb.train() Booster, evals_result is passed as a dict callback
evals_result = {}

# --- Re-train with evals_result dict to capture history ---
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
    lgb.record_evaluation(evals_result),   # <-- this captures the history
]

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_test],
    valid_names=["train", "test"],
    callbacks=callbacks,
)

# Now plot from evals_result dict
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(evals_result["train"]["rmse"], label="Training RMSE",  linewidth=2.5)
ax.plot(evals_result["test"]["rmse"],  label="Testing RMSE",
        linewidth=2.5, linestyle="--")
ax.axvline(model.best_iteration, color="grey", linestyle=":",
           linewidth=1.5, label=f"Best iter ({model.best_iteration})")
ax.set_xlabel("Boosting Round")
ax.set_ylabel("RMSE")
ax.set_title("LightGBM Learning Curve")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, "loss_curve.png"),
            dpi=350, bbox_inches="tight")
plt.close(fig)
print("Saved: loss_curve.png")
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

for ax, (pred, obs, label) in zip(axes, [
    (y_train_pred, y_train, "(a)"),
    (y_test_pred,  y_test,  "(b)"),
]):
    residuals = obs - pred
    ax.scatter(pred, residuals, alpha=0.25, s=6,
               color="#d7191c", edgecolors="none", rasterized=True)
    ax.axhline(0, color="black", linewidth=2, linestyle="--")
    ax.set_xlabel("Predicted CDI")
    ax.set_ylabel("Residuals (Observed - Predicted)")
    ax.set_title(label)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.text(0.98, 0.95, f"Mean residual = {residuals.mean():.4f}",
            transform=ax.transAxes, fontsize=FS - 4,
            ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="grey", alpha=0.85))

plt.tight_layout()
fig.savefig(os.path.join(output_dir, "residuals_combined.png"),
            dpi=300, bbox_inches="tight")
plt.close(fig)
print("Saved: residuals_combined.png")

# =============================================================================
# FINAL REPORT
# =============================================================================

print("\n" + "=" * 70)
print("RESULTS SAVED TO:", output_dir)
print("=" * 70)
print(f"\n  Features used  : {FEATURE_COLS}")
print(f"  Best iteration : {model.best_iteration}")
print(f"\n  Training -> RMSE={tr[1]:.4f}  MAE={tr[2]:.4f}  "
      f"R2={tr[0]:.4f}  NSE={tr[3]:.4f}  PBIAS={tr[4]:.2f}%")
print(f"  Testing  -> RMSE={te[1]:.4f}  MAE={te[2]:.4f}  "
      f"R2={te[0]:.4f}  NSE={te[3]:.4f}  PBIAS={te[4]:.2f}%")
print(f"\n  Overfit Gap (Train R2 - Test R2) : {gap:.4f}")
print(f"  Common period used : {common_keys[0]} to {common_keys[-1]}")
print(f"  Total pooled samples: {len(precip_all):,}")
print("\nFinished Successfully.")

LightGBM FULL-PIXEL MODEL — CHIRPS + ERA5 -> CDI

  CHIRPS months : 540
  ERA5   months : 540
  CDI    months : 311
  COMMON months : 311
  Period        : 2000_02 to 2025_12

Loading and pooling all pixels across common months ...
  Processed 50/311 months ...
  Processed 100/311 months ...
  Processed 150/311 months ...
  Processed 200/311 months ...
  Processed 250/311 months ...
  Processed 300/311 months ...
  Processed 311/311 months ...

  Total pooled samples: 42,607
          precip      tmean      month        cdi
count  42607.000  42607.000  42607.000  42607.000
mean      81.402     21.770      6.518     -0.591
std      106.393      4.433      3.443      1.749
min        0.000     11.433      1.000     -3.540
25%        6.525     18.453      4.000     -2.116
50%       28.136     21.377      7.000     -1.151
75%      125.229     24.717     10.000      1.017
max      555.889     35.231     12.000      3.815

  X shape : (42607, 3)  (features: ['precip', 'tmean', 'month'])
  y 

## LightGBM-futer CDI

In [9]:
# =============================================================================
# FUTURE CDI PROJECTION (2030–2050) — LightGBM — NEX-GDDP-CMIP6
# =============================================================================
# SCENARIOS : ssp245, ssp585
# MODELS    : PR -> GFDL-CM4 | TA -> INM-CM5-0
# FEATURES  : precip, tmean, month_of_year (same as training)
# =============================================================================

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable

import lightgbm as lgb
from sklearn.preprocessing import StandardScaler

# =============================================================================
# PATHS
# =============================================================================

BASE_DIR   = r"C:\Users\gerii\Documents\projects\tekeze"
PR_MODEL   = "GFDL-CM4"
TA_MODEL   = "INM-CM5-0"

PR_DIR     = os.path.join(BASE_DIR, "NEX-GDDP-CMIP6", "pr_corrected", PR_MODEL)
TA_DIR     = os.path.join(BASE_DIR, "NEX-GDDP-CMIP6", "ta_corrected", TA_MODEL)

# for scaler refit (same historical data as training)
CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_resampled")
ERA5_DIR   = os.path.join(BASE_DIR, "ERA5_resampled")
CDI_DIR    = os.path.join(BASE_DIR, "indices", "CDI_resampled")

MODEL_PATH = os.path.join(BASE_DIR, "lgbm_fullpixel_results_v1",
                          "lgbm_model_fullpixel_v1.txt")

OUT_DIR    = os.path.join(BASE_DIR, "NEX-GDDP-CMIP6",
                          "CDI_future_lgbm")

SCENARIOS  = ["ssp245", "ssp585"]
START_YEAR = 2030
END_YEAR   = 2050
NODATA     = -9999.0

os.makedirs(OUT_DIR, exist_ok=True)

# =============================================================================
# GLOBAL FONT SIZE
# =============================================================================

FS = 16
plt.rcParams.update({
    "font.size":        FS,
    "axes.titlesize":   FS,
    "axes.labelsize":   FS,
    "xtick.labelsize":  FS - 2,
    "ytick.labelsize":  FS - 2,
    "legend.fontsize":  FS - 2,
    "font.family":      "serif",
    "axes.linewidth":   1.2,
})

# =============================================================================
# CDI COLORMAP — standard drought severity classes
# =============================================================================

CDI_BOUNDS = [-3.5, -2.0, -1.5, -1.0, -0.5,  0.5,  1.0,  1.5,  2.0,  3.5]
CDI_COLORS = [
    "#730000",  # Exceptional drought  < -2.0
    "#E60000",  # Extreme drought     -2.0 to -1.5
    "#FFAA00",  # Severe drought      -1.5 to -1.0
    "#FCD37F",  # Moderate drought    -1.0 to -0.5
    "#FFFF00",  # Abnormally dry      -0.5 to  0.5
    "#FFFFFF",  # Near normal          0.5 to  1.0
    "#00AAFF",  # Abnormally wet       1.0 to  1.5
    "#0000FF",  # Moderately wet       1.5 to  2.0
    "#00008B",  # Extremely wet       > 2.0
]
cdi_cmap = mcolors.ListedColormap(CDI_COLORS)
cdi_norm = mcolors.BoundaryNorm(CDI_BOUNDS, cdi_cmap.N)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_raster(path):
    with rasterio.open(path) as src:
        arr     = src.read(1).astype(np.float32)
        profile = src.profile
        nd      = src.nodata
    if nd is not None:
        arr[arr == nd] = np.nan
    arr[arr <= -9999] = np.nan
    return arr, profile


def save_raster(path, arr, profile):
    out = arr.copy()
    out[np.isnan(out)] = NODATA
    profile = profile.copy()
    profile.update(dtype="float32", count=1, nodata=NODATA, compress="lzw")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out.astype(np.float32), 1)


def tif_keys(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.tif")))
    return set(os.path.basename(x).replace(".tif", "") for x in files)

# =============================================================================
# STEP 1: LOAD LightGBM MODEL
# =============================================================================

print("=" * 70)
print("FUTURE CDI PROJECTION — LightGBM — 2030 to 2050")
print("=" * 70)

print(f"\nLoading LightGBM model from:\n  {MODEL_PATH}")
model = lgb.Booster(model_file=MODEL_PATH)
print(f"  Best iteration : {model.best_iteration}")
print("Model loaded successfully.")

# =============================================================================
# STEP 2: REFIT SCALER ON HISTORICAL CHIRPS + ERA5 + MONTH
# =============================================================================
# Must refit with the SAME 3-feature order as training:
# [precip, tmean, month_of_year]

print("\nRefitting StandardScaler on historical CHIRPS + ERA5 data ...")

common_keys = sorted(tif_keys(CHIRPS_DIR) & tif_keys(ERA5_DIR) & tif_keys(CDI_DIR))
print(f"  Historical months found : {len(common_keys)}")
print(f"  Period                  : {common_keys[0]} to {common_keys[-1]}")

P_list, T_list, M_list = [], [], []

for i, key in enumerate(common_keys):
    month_num  = int(key.split("_")[1])
    p_arr, _   = load_raster(os.path.join(CHIRPS_DIR, f"{key}.tif"))
    t_arr, _   = load_raster(os.path.join(ERA5_DIR,   f"{key}.tif"))

    if p_arr.shape != t_arr.shape:
        continue

    p_flat = p_arr.flatten()
    t_flat = t_arr.flatten()
    valid  = (~np.isnan(p_flat)) & (~np.isnan(t_flat))
    n      = valid.sum()
    if n == 0:
        continue

    P_list.append(p_flat[valid])
    T_list.append(t_flat[valid])
    M_list.append(np.full(n, month_num, dtype=np.float32))

    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(common_keys)} months ...")

X_hist = np.column_stack([
    np.concatenate(P_list),
    np.concatenate(T_list),
    np.concatenate(M_list),
])

scaler = StandardScaler()
scaler.fit(X_hist)

print(f"  Scaler fitted on {len(X_hist):,} samples.")
print(f"  Feature means  : precip={scaler.mean_[0]:.3f}  "
      f"tmean={scaler.mean_[1]:.3f}  month={scaler.mean_[2]:.3f}")
print(f"  Feature scales : precip={scaler.scale_[0]:.3f}  "
      f"tmean={scaler.scale_[1]:.3f}  month={scaler.scale_[2]:.3f}")

del X_hist, P_list, T_list, M_list

# =============================================================================
# STEP 3: BUILD FUTURE MONTH LIST 2030–2050
# =============================================================================

future_months = []
for y in range(START_YEAR, END_YEAR + 1):
    for m in range(1, 13):
        future_months.append(f"{y}_{m:02d}")

print(f"\n  Total future months to predict : {len(future_months)}")
print(f"  Period                         : "
      f"{future_months[0]} to {future_months[-1]}")

# =============================================================================
# STEP 4: LOOP SCENARIOS -> MONTHS -> PREDICT -> SAVE
# =============================================================================

for scenario in SCENARIOS:

    print("\n" + "=" * 70)
    print(f"SCENARIO : {scenario.upper()}")
    print("=" * 70)

    pr_dir  = os.path.join(PR_DIR, scenario)
    ta_dir  = os.path.join(TA_DIR, scenario)
    out_dir = os.path.join(OUT_DIR, scenario)
    map_dir = os.path.join(OUT_DIR, f"{scenario}_maps")

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(map_dir, exist_ok=True)

    saved, missing, empty = 0, 0, 0
    summary_rows = []

    for key in future_months:
        year, month = key.split("_")
        month_num   = int(month)

        pr_file  = os.path.join(pr_dir, f"{key}.tif")
        ta_file  = os.path.join(ta_dir, f"{key}.tif")
        out_file = os.path.join(out_dir, f"CDI_lgbm_{scenario}_{key}.tif")

        # check files exist
        if not os.path.exists(pr_file):
            print(f"  MISSING PR : {key}")
            missing += 1
            continue
        if not os.path.exists(ta_file):
            print(f"  MISSING TA : {key}")
            missing += 1
            continue

        p_arr, profile = load_raster(pr_file)
        t_arr, _       = load_raster(ta_file)

        if p_arr.shape != t_arr.shape:
            print(f"  Shape mismatch : {key} — skipping")
            continue

        rows, cols = p_arr.shape
        p_flat     = p_arr.flatten()
        t_flat     = t_arr.flatten()
        valid      = (~np.isnan(p_flat)) & (~np.isnan(t_flat))
        n_valid    = valid.sum()

        if n_valid == 0:
            print(f"  No valid pixels : {key}")
            empty += 1
            continue

        # ---- build feature matrix [precip, tmean, month] ----
        X = np.column_stack([
            p_flat[valid],
            t_flat[valid],
            np.full(n_valid, month_num, dtype=np.float32),
        ])

        X_sc = scaler.transform(X)
        X_sc = np.clip(X_sc, -6, 6)   # same safety clip as CNN script

        # ---- predict ----
        cdi_pred = model.predict(X_sc,
                                 num_iteration=model.best_iteration)

        # ---- reconstruct full raster ----
        cdi_full        = np.full(rows * cols, np.nan, dtype=np.float32)
        cdi_full[valid] = cdi_pred.astype(np.float32)
        cdi_map         = cdi_full.reshape(rows, cols)

        # ---- save GeoTIFF ----
        save_raster(out_file, cdi_map, profile)
        saved += 1

        # ---- spatial statistics ----
        valid_cdi = cdi_map[~np.isnan(cdi_map)]
        summary_rows.append({
            "Scenario":    scenario,
            "Month":       key,
            "Year":        int(year),
            "MonthNum":    month_num,
            "CDI_mean":    round(float(valid_cdi.mean()),   4),
            "CDI_std":     round(float(valid_cdi.std()),    4),
            "CDI_min":     round(float(valid_cdi.min()),    4),
            "CDI_max":     round(float(valid_cdi.max()),    4),
            "CDI_median":  round(float(np.median(valid_cdi)), 4),
            "Pct_drought": round(float((valid_cdi < -0.5).mean() * 100), 2),
            "Pct_severe":  round(float((valid_cdi < -1.5).mean() * 100), 2),
            "Pct_extreme": round(float((valid_cdi < -2.0).mean() * 100), 2),
        })

        # ---- monthly CDI map ----
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cdi_map, cmap=cdi_cmap, norm=cdi_norm,
                       interpolation="nearest")
        cbar = fig.colorbar(ScalarMappable(norm=cdi_norm, cmap=cdi_cmap),
                            ax=ax, fraction=0.035, pad=0.04)
        cbar.set_label("CDI", fontsize=FS)
        cbar.set_ticks([-3.0, -2.0, -1.5, -1.0, -0.5,
                         0.5,  1.0,  1.5,  2.0,  3.0])
        cbar.ax.tick_params(labelsize=FS - 4)
        ax.set_title(f"Predicted CDI — {scenario.upper()} — {year}/{month}",
                     fontsize=FS)
        ax.set_xlabel("Column (pixel)")
        ax.set_ylabel("Row (pixel)")
        stats_txt = (
            f"Mean  = {valid_cdi.mean():.2f}\n"
            f"Std   = {valid_cdi.std():.2f}\n"
            f"% Drought = {(valid_cdi < -0.5).mean()*100:.1f}%"
        )
        ax.text(0.02, 0.98, stats_txt, transform=ax.transAxes,
                fontsize=FS - 5, verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                          edgecolor="grey", alpha=0.85))
        plt.tight_layout()
        fig.savefig(os.path.join(map_dir, f"CDI_map_{scenario}_{key}.png"),
                    dpi=200, bbox_inches="tight")
        plt.close(fig)

        print(f"  {key} | mean={valid_cdi.mean():.3f} "
              f"drought%={(valid_cdi < -0.5).mean()*100:.1f}% "
              f"| saved")

    # ---------------------------------------------------------------
    # SAVE SUMMARY CSV PER SCENARIO
    # ---------------------------------------------------------------
    df_sum  = pd.DataFrame(summary_rows)
    csv_out = os.path.join(OUT_DIR, f"future_cdi_summary_{scenario}.csv")
    df_sum.to_csv(csv_out, index=False)
    print(f"\n  Summary CSV saved: {csv_out}")

    # ---------------------------------------------------------------
    # BASIN-WIDE TIME SERIES PLOT PER SCENARIO
    # ---------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(18, 6))
    x     = range(len(df_sum))
    means = df_sum["CDI_mean"].values

    bar_colors = ["#E60000" if v < -1.0 else
                  "#FFAA00" if v < -0.5 else
                  "#00AAFF" if v >  0.5 else
                  "#AAAAAA" for v in means]

    ax.bar(x, means, color=bar_colors, alpha=0.85, edgecolor="none")
    ax.axhline(0,    color="black",   linewidth=1.2, linestyle="--")
    ax.axhline(-0.5, color="#FFAA00", linewidth=1.0, linestyle=":",
               label="Drought threshold (−0.5)")
    ax.axhline(-1.5, color="#E60000", linewidth=1.0, linestyle=":",
               label="Severe threshold (−1.5)")

    # x-tick: show only Jan of each year to avoid crowding
    tick_pos   = [j for j, k in enumerate(df_sum["Month"])
                  if k.endswith("_01")]
    tick_labels = [k.split("_")[0] for k in df_sum["Month"]
                   if k.endswith("_01")]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_labels, rotation=45, ha="right")

    ax.set_ylabel("Basin-mean Predicted CDI")
    ax.set_title(f"Future Basin-Wide CDI — LightGBM — {scenario.upper()} "
                 f"({START_YEAR}–{END_YEAR})")
    ax.legend(fontsize=FS - 4, loc="upper right")
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    ts_path = os.path.join(OUT_DIR,
                           f"future_cdi_timeseries_{scenario}.png")
    fig.savefig(ts_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Time series plot saved: {ts_path}")

    print(f"\n  {scenario.upper()} COMPLETE — "
          f"Saved={saved}  Missing={missing}  Empty={empty}")

# =============================================================================
# FINAL REPORT
# =============================================================================

print("\n" + "=" * 70)
print("ALL SCENARIOS COMPLETE")
print("=" * 70)
print(f"  Scenarios     : {SCENARIOS}")
print(f"  Period        : {START_YEAR} to {END_YEAR}")
print(f"  PR model      : {PR_MODEL}")
print(f"  TA model      : {TA_MODEL}")
print(f"  Output dir    : {OUT_DIR}")
print("\nFinished Successfully.")

FUTURE CDI PROJECTION — LightGBM — 2030 to 2050

Loading LightGBM model from:
  C:\Users\gerii\Documents\projects\tekeze\lgbm_fullpixel_results_v1\lgbm_model_fullpixel_v1.txt
  Best iteration : -1
Model loaded successfully.

Refitting StandardScaler on historical CHIRPS + ERA5 data ...
  Historical months found : 311
  Period                  : 2000_02 to 2025_12
  Processed 50/311 months ...
  Processed 100/311 months ...
  Processed 150/311 months ...
  Processed 200/311 months ...
  Processed 250/311 months ...
  Processed 300/311 months ...
  Scaler fitted on 45,406 samples.
  Feature means  : precip=80.695  tmean=21.658  month=6.518
  Feature scales : precip=105.669  tmean=4.552  month=3.443

  Total future months to predict : 252
  Period                         : 2030_01 to 2050_12

SCENARIO : SSP245
  2030_01 | mean=-2.074 drought%=100.0% | saved
  2030_02 | mean=-2.504 drought%=100.0% | saved
  2030_03 | mean=-2.445 drought%=100.0% | saved
  2030_04 | mean=-2.411 drought%=100.

## visualization